# Chapter 8 — Task Decomposition and Orchestration · Homework (self-test)

**Goal:** check that you can pick a decomposition shape, guard the seams of an orchestration in code (plan coverage, bounded fan-out, capped evaluator loops, independent review), and reason about handoffs and hosted multi-agent options, without looking at the answers.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 1.5–2 hours
- **Estimated API cost:** $0 for the graded parts (they run offline on scripted models and stubs). The optional live bonus exercise costs about $0.01 with the default `gpt-6-luna`, and runs only if you set `RUN_LIVE_BONUS = True`.
- **Prerequisites:** the chapter README and the practice notebook.

### How to use this notebook

1. Run the setup cells.
2. **Part A:** answer 10 scenario questions in the `answers` cell, then run the grader. The answer key is stored as hashes, so you can't read it by accident.
3. **Part B:** implement each `TODO`, then run its checker. Checkers never crash the notebook; they print ✓ or a hint. Exercises 3, 4 and 6 are `async`: their checker cells use `await`.
4. **Part C:** write a short architecture answer, then compare it with the rubric.
5. **Scorecard:** quiz and exercises 1–5 weigh 50% each. The pass mark is **72%**, the same bar as the Claude path. Exercise 6 is a bonus.

Solutions are at the very bottom, inside collapsed blocks. Attempt everything first.

In [ ]:
import asyncio
import copy
import hashlib
import json
import os
import re
import time
from dataclasses import dataclass
from typing import Any, Awaitable, Callable, Literal

from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, set_tracing_disabled
from agents.testing import ModelCall, ModelStep, ScriptedModel, UnexpectedModelCall, assistant_message

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
HAVE_REAL_KEY = bool(os.getenv("OPENAI_API_KEY")) and not os.getenv("NBCHECK")  # NBCHECK: this repo's checker
set_tracing_disabled(not HAVE_REAL_KEY)
print(f"MODEL={MODEL} | live API available: {HAVE_REAL_KEY}")

In [ ]:
# Only the optional live bonus (exercise 6) calls the API, and only when you opt in here (~$0.01 on MODEL).
# With the default False, "Run All" stays offline and costs nothing, with or without a key.
RUN_LIVE_BONUS = False
if RUN_LIVE_BONUS:
    assert HAVE_REAL_KEY, "Set OPENAI_API_KEY in .env (see 00-prerequisites)"

In [ ]:
# Shared fixtures and grading helpers. Run this cell, do not edit it.
class Subtask(BaseModel):
    id: str
    facet: str
    question: str
    context: str


class Plan(BaseModel):
    subtasks: list[Subtask]


class Review(BaseModel):
    rationale: str
    verdict: Literal["pass", "revise"]
    feedback: list[str]


REQUIRED_FACETS = {"security", "data", "cost", "operations"}


@dataclass
class PlanReport:
    missing_facets: list[str]     # sorted, lowercase required facets that no subtask covers
    duplicate_ids: list[str]      # sorted subtask ids that appear more than once (each id listed once)
    empty_context_ids: list[str]  # sorted ids whose context is empty or only whitespace
    too_many: bool                # more subtasks than max_subtasks

    @property
    def ok(self) -> bool:
        return not (self.missing_facets or self.duplicate_ids or self.empty_context_ids or self.too_many)


class TransientWorkerError(Exception):
    """A rate limit or timeout: worth one more try."""


def say(text: str):
    return assistant_message(text)


def policy_model(policy: Callable, max_calls: int = 10) -> ScriptedModel:
    """A scripted model that answers each call with policy(call), at most max_calls times."""
    return ScriptedModel([ModelStep.respond(policy) for _ in range(max_calls)])


def user_text(call: ModelCall) -> str:
    items = call.input if isinstance(call.input, list) else [{"role": "user", "content": call.input}]
    for item in reversed(items):
        if isinstance(item, dict) and item.get("role") == "user":
            content = item.get("content")
            return content if isinstance(content, str) else " ".join(p.get("text", "") for p in content or [])
    return ""


EX_RESULTS: dict[int, bool] = {}


def _record(n: int, ok: bool, message: str) -> None:
    EX_RESULTS[n] = ok
    print(message)


def _run_check(n: int, checks: Callable[[], None]) -> None:
    """Run one exercise's asserts and record the result. Never raises."""
    try:
        checks()
    except NotImplementedError:
        _record(n, False, f"✗ Exercise {n}: not implemented yet. Replace the TODO and rerun this cell.")
    except AssertionError as err:
        _record(n, False, f"✗ Exercise {n} not passed yet: {err}")
    except Exception as err:  # a bug in the solution must not crash the notebook
        _record(n, False, f"✗ Exercise {n} raised {type(err).__name__}: {err}")
    else:
        _record(n, True, f"✓ Exercise {n} passed")


async def _run_check_async(n: int, checks: Callable[[], Awaitable[None]]) -> None:
    try:
        await checks()
    except NotImplementedError:
        _record(n, False, f"✗ Exercise {n}: not implemented yet. Replace the TODO and rerun this cell.")
    except AssertionError as err:
        _record(n, False, f"✗ Exercise {n} not passed yet: {err}")
    except UnexpectedModelCall:
        _record(n, False, f"✗ Exercise {n}: a scripted model was called more often than the task allows. Is your loop capped?")
    except Exception as err:
        _record(n, False, f"✗ Exercise {n} raised {type(err).__name__}: {err}")
    else:
        _record(n, True, f"✓ Exercise {n} passed")


print("fixtures ready")

## Part A — Quiz

Ten exam-style scenarios. Exactly one answer is defensible per question.

**Q1.** Every incoming contract goes through the same steps: extract parties and dates, validate the dates against the CRM, enrich with the account owner, and post a summary to the deal record. Each step depends on the previous one. A developer proposes a manager agent with four `Agent.as_tool()` workers that "decides the best order each time". What should the architect recommend?

- A) A fixed pipeline: one agent per step chained in code, with structured `output_type` gates between steps
- B) The manager agent, because it can adapt the order to each contract
- C) A triage agent that hands off from one step specialist to the next
- D) Responses API Multi-agent with one subagent per step

**Q2.** A review manager emits three `as_tool()` worker calls in one turn. All workers call an internal API that allows at most **2** concurrent requests, and the team now sees 429 errors. They want to keep as much parallelism as the backend allows. What is the best fix?

- A) Set `ModelSettings(parallel_tool_calls=False)` on the manager
- B) Turn the workers into handoffs, since only one handoff runs per turn
- C) Run the manager with `RunConfig(tool_execution=ToolExecutionConfig(max_function_tool_concurrency=2))`
- D) Lower `max_turns` on the manager run to 2

**Q3.** In an evaluator-optimizer loop for release notes, the evaluator returns `verdict="revise"` on round 1 for almost every draft, including drafts that human reviewers rate as good. Most runs end at the round cap. What does this indicate, and what should you do?

- A) The cap is too low; raise `max_rounds` to 10
- B) The rubric is too strict or contradicts the drafter's instructions; compare the two, check the evaluator against human labels, and fix the criteria
- C) The evaluator lacks context; give it the drafter's full conversation
- D) Use a smaller evaluator model so it is more lenient

**Q4.** A manager agent knows the customer's plan, seat count and region from the conversation. It calls a pricing worker built with `Agent.as_tool()` with the input "quote the upgrade", and the worker keeps quoting the wrong plan. The manager must keep control and merge several workers' answers. What is the root cause and the fix?

- A) The worker model is too weak; switch it to `gpt-6.1-sol`
- B) Make pricing a handoff target so it sees the conversation history
- C) Raise the worker's `max_turns` so it can ask follow-up questions
- D) An `as_tool()` worker sees only its tool input; pass the facts explicitly, for example with a structured `parameters=` model that has plan, seats and region fields

**Q5.** A Responses API Multi-agent request sets `multi_agent={"enabled": True, "max_concurrent_subagents": 3}`. The root spawns two reviewers, and each reviewer wants to spawn a tester. How many subagents can be active at the same time?

- A) At most 3 across the whole tree (children, grandchildren and deeper), not counting `/root`
- B) 3 per parent agent, so up to 6
- C) 3 including `/root`, so at most 2 subagents
- D) The limit applies only to direct children of `/root`; testers are unlimited

**Q6.** An Agents API session enables `multi_agent` so a coordinator can delegate to subagents. The team wants each subagent to call the company's `get_invoice` **function tool**. What is accurate?

- A) Add `get_invoice` to each subagent's tool list in the session request
- B) Subagents in the Agents API do not support function tools; they inherit MCP tools, web search settings and the environment. Expose the capability through an MCP server configured on the agent, keep it with the coordinator, or orchestrate with the Agents SDK
- C) Execute the subagents' function calls with `response.inject` over WebSocket
- D) Set `max_concurrent_subagents=0` so the coordinator handles function calls

**Q7.** A coding agent writes patches that are merged after the same agent session "reviews" them with extended reasoning. Bugs slip through. The team wants AI review in CI. Which design is best?

- A) Keep the same-session review but raise `reasoning.effort` to `xhigh`
- B) Continue the authoring conversation with `previous_response_id` and ask for a review, so the reviewer has full context
- C) Add a separate review step, such as `codex review --base main` in CI or a fresh `Runner.run` that gets only the diff and the `## Code Review Rules`, and keep tests, branch protection and required approvals
- D) Replace required human approvals with `@codex review`, since it flags P0 and P1 issues

**Q8.** An orchestration plans up to 6 subtasks, runs each worker with `max_turns=10`, and adds up to 3 evaluator rounds. Finance wants a hard ceiling on model calls and tokens per job. What should you implement?

- A) Lower `max_turns` on every `Runner.run` to 3
- B) Set `max_budget_usd` in `RunConfig`, as in the Claude Agent SDK
- C) Rely on the project's rate limits to cap spend
- D) An orchestration-wide budget in code: before each `Runner.run`, reserve its worst case (`max_turns` requests) and refuse to start it if that could cross the ceiling; after it, add the real `result.context_wrapper.usage`; cap output tokens per call

**Q9.** A security review fans out one worker per changed file (12 files). One worker keeps timing out. What should the orchestrator do?

- A) Approve based on the 11 completed reviews; one file is a small gap
- B) Fail the whole job and discard the 11 completed reviews
- C) Retry the transient failure (bounded); if it still fails, mark the file `unreviewed`, keep the other 11 results, and block an "approve" verdict
- D) Ask the synthesizer to infer the likely findings for the missing file

**Q10.** Three Codex subagents implement parts of a feature at the same time in one checkout and keep overwriting each other's changes. What is the best fix?

- A) Raise `agents.max_concurrent_threads_per_session` so they finish before they collide
- B) Give each parallel writer its own worktree (and branch) with a single merge owner, and keep parallel subagents mainly for read-heavy work such as exploration, tests and triage
- C) Ask each subagent in `AGENTS.md` to be careful not to overwrite others' changes
- D) Switch the subagents to a cheaper model so they finish faster

In [ ]:
# Fill in one letter per question ("A"-"D"). Leave None to skip.
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
}

In [ ]:
_KEY = {
    "q1": "d01e0d2123c2608d1604388f402be6311fec6f81aacc56159f9974fc375704dd",
    "q2": "c85dcf1566dc5c0487a6fe1611c41ba32751e8080b552a419521a39c7fee21b2",
    "q3": "74fe5019edcd781025df8689d0e86f17392990e8335c3ef475852d2e1207b9f8",
    "q4": "b634a8333266441f30307b2d3392d77b7b3bcf69bc053cabed2c65d2671fc12e",
    "q5": "efeafc86344da0b52bbaa7783b6621595e70274062976345ce6c533b9322877c",
    "q6": "621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40",
    "q7": "242d38a9a6e8adda29d627e4ede1ef3c7395ee0112db33f8f0b06b45d9a66887",
    "q8": "85b91a1895880cedf03bb98aa9ebfb3c730c838ec5d8f59ec2d1d86bcb936ced",
    "q9": "e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7",
    "q10": "c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3",
}  # sha256 of "qid:LETTER", so the answers are not readable here


def grade_quiz(answers: dict) -> float:
    correct = 0
    for qid, digest in _KEY.items():
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else ""
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        correct += ok
        print(f"{qid:>3}: {'✓' if ok else '✗'}  (your answer: {letter or '-'})")
    score = correct / len(_KEY)
    print(f"Quiz score: {correct}/{len(_KEY)} = {score:.0%}")
    return score


quiz_score = grade_quiz(answers)

## Part B — Coding exercises

Exercises 1–5 are offline and graded. Exercise 6 is an optional live bonus with an offline dry run.

### Exercise 1 — Pick the decomposition shape

For each workload, choose one label from `SHAPE_LABELS`. [Theory: 8.1](README.md#81-workflows-vs-agents-choosing-a-decomposition-shape)

| id | Workload |
|---|---|
| `w1` | Classify the sentiment of one product review as positive, neutral or negative. |
| `w2` | Every contract: extract parties and dates → validate the dates against the CRM → write a summary. Same steps each time; each needs the previous result. |
| `w3` | Incoming emails are sales leads, support requests or invoices; each type needs a different prompt, tool set and model. |
| `w4` | Summarize 40 independent customer interviews for a quarterly report; latency matters. |
| `w5` | Investigate why the nightly data pipeline produced duplicate rows; which components to inspect depends on what the first checks reveal. |
| `w6` | Write a regulator-facing incident notice that must satisfy a 6-point compliance rubric; specific reviewer feedback reliably improves drafts. |

In [ ]:
SHAPE_LABELS = {"single_call", "fixed_pipeline", "routing", "parallelization", "orchestrator_workers", "evaluator_optimizer"}

# TODO: replace each None with one label from SHAPE_LABELS
SHAPES = {"w1": None, "w2": None, "w3": None, "w4": None, "w5": None, "w6": None}

In [ ]:
_SHAPE_KEY = {
    "w1": "fbf9ef11ce1ebca0e0e65172308f10eb9c6b46d55e6eda94dbc90469ba02e958",
    "w2": "a847ac4d5974ec974d75f62fe893c6f567172b678258793b5202b1b81eb56ed6",
    "w3": "a8610ededc9991defc1a8250035f144137d693caf4be428ebf780915ecb45e5b",
    "w4": "45ea993c7c41a832c95f523f11dc0e82d0a5b1514b11a5fec067df04c33ee776",
    "w5": "f3675369569279e698b09c958a834da87629e5b8340fad835365a83add0e5b77",
    "w6": "daba37c234924a8941b9974e240a0460d1ca67ee86a43a7c6c4bb1195f846998",
}
_SHAPE_HINTS = {
    "w1": "Does one focused call already meet the outcome?",
    "w2": "Are the steps known up front and dependent on each other? Then code should own the order.",
    "w3": "Is the decision a choice among known categories that need different prompts, tools or models?",
    "w4": "Are the parts independent of each other, so wall-clock time can be the slowest part instead of the sum?",
    "w5": "Are the subtasks unknown until intermediate results come in?",
    "w6": "Is there an explicit rubric, and does feedback improve the result?",
}


def check_ex1():
    def checks():
        assert isinstance(SHAPES, dict) and set(SHAPES) == set(_SHAPE_KEY), "SHAPES needs exactly the keys w1..w6"
        unknown = {k: v for k, v in SHAPES.items() if v is not None and v not in SHAPE_LABELS}
        assert not unknown, f"use labels from SHAPE_LABELS, got {unknown}"
        if all(v is None for v in SHAPES.values()):
            raise NotImplementedError
        wrong = [k for k, v in SHAPES.items()
                 if v is None or hashlib.sha256(f"{k}:{v}".encode()).hexdigest() != _SHAPE_KEY[k]]
        assert not wrong, f"{len(wrong)} workload(s) not right yet. Hints: " + " | ".join(f"{k}: {_SHAPE_HINTS[k]}" for k in wrong)
    _run_check(1, checks)


check_ex1()

### Exercise 2 — A plan coverage validator

Implement `validate_plan(plan, required_facets, max_subtasks=6)` and return a `PlanReport` (defined in the fixtures cell). This is the code-side check that catches **over-narrow decomposition** before any worker runs. [Theory: 8.5](README.md#85-dynamic-decomposition-orchestrator-workers)

- `missing_facets`: required facets that no subtask covers, **sorted** and **lowercase**. Compare facets after `strip().lower()` on both sides, so `"Security "` covers `"security"`.
- `duplicate_ids`: subtask ids that appear more than once, sorted, each id listed once.
- `empty_context_ids`: ids of subtasks whose `context` is empty or whitespace (a worker sees nothing else), sorted.
- `too_many`: `True` when the plan has more than `max_subtasks` subtasks.
- Do not mutate the plan.

In [ ]:
def validate_plan(plan: Plan, required_facets: set[str], max_subtasks: int = 6) -> PlanReport:
    # TODO: return a PlanReport
    raise NotImplementedError

In [ ]:
def check_ex2():
    def st(i, facet, context="facts"):
        return Subtask(id=i, facet=facet, question="q", context=context)

    def checks():
        good = Plan(subtasks=[st("s1", "security"), st("s2", "data"), st("s3", "cost"), st("s4", "operations")])
        r = validate_plan(good, REQUIRED_FACETS)
        assert isinstance(r, PlanReport), "return a PlanReport"
        assert r.ok and r.missing_facets == [] and r.duplicate_ids == [] and r.empty_context_ids == [] and r.too_many is False, \
            f"a complete plan should be ok, got {r}"
        narrow = Plan(subtasks=[st("s1", "security"), st("s2", "data")])
        r = validate_plan(narrow, REQUIRED_FACETS)
        assert r.missing_facets == ["cost", "operations"], f"missing facets must be sorted: expected ['cost', 'operations'], got {r.missing_facets}"
        assert not r.ok
        messy = Plan(subtasks=[st("s1", "Security "), st("s2", "DATA"), st("s3", "cost"), st("s4", " Operations")])
        assert validate_plan(messy, REQUIRED_FACETS).missing_facets == [], "normalize facets with strip().lower()"
        assert validate_plan(good, {"Security", " COST"}).missing_facets == [], "normalize the required facets too"
        assert validate_plan(narrow, {"Cost", "security"}).missing_facets == ["cost"], "report missing facets in lowercase"
        dup = Plan(subtasks=[st("s1", "security"), st("s1", "data"), st("s1", "cost"), st("s2", "operations"), st("s2", "cost")])
        assert validate_plan(dup, REQUIRED_FACETS).duplicate_ids == ["s1", "s2"], "list each duplicated id once, sorted"
        empty = Plan(subtasks=[st("s2", "security", "  "), st("s1", "data", ""), st("s3", "cost"), st("s4", "operations")])
        assert validate_plan(empty, REQUIRED_FACETS).empty_context_ids == ["s1", "s2"], "flag empty/whitespace contexts, sorted"
        big = Plan(subtasks=[st(f"s{i}", f) for i, f in enumerate(["security", "data", "cost", "operations", "cost", "data", "security"])])
        assert validate_plan(big, REQUIRED_FACETS).too_many is True, "7 subtasks > max_subtasks=6"
        assert validate_plan(big, REQUIRED_FACETS, max_subtasks=7).too_many is False, "respect the max_subtasks argument"
        snapshot = narrow.model_dump()
        validate_plan(narrow, REQUIRED_FACETS)
        assert narrow.model_dump() == snapshot, "do not mutate the plan"
    _run_check(2, checks)


check_ex2()

### Exercise 3 — Bounded fan-out with partial failures

Implement `bounded_fan_out(worker, inputs, limit=3, retries=1)`. `worker` is any `async` callable `worker(text) -> str` (in production: a function that awaits `Runner.run`). [Theory: 8.4](README.md#84-parallelization-fan-out-voting-and-parallel-tool-calls)

- Return one `(status, value)` tuple per input, **in input order**: `("ok", result)` or `("failed", "<ExceptionClassName>")`.
- At most `limit` worker calls may be in flight at once.
- Retry a slot up to `retries` more times, but **only** on `TransientWorkerError`. Any other exception fails the slot immediately (retrying a bug wastes calls).
- One failed slot must never cancel or discard the others.

In [ ]:
async def bounded_fan_out(worker: Callable[[str], Awaitable[str]], inputs: list[str],
                          limit: int = 3, retries: int = 1) -> list[tuple[str, str]]:
    # TODO
    raise NotImplementedError

In [ ]:
class _StubWorker:
    """ok-*: succeeds | flaky: transient error once, then succeeds | down: always transient | bug: ValueError."""

    def __init__(self, delay: float = 0.05) -> None:
        self.delay, self.in_flight, self.peak, self.calls = delay, 0, 0, {}

    async def __call__(self, text: str) -> str:
        self.calls[text] = self.calls.get(text, 0) + 1
        self.in_flight += 1
        self.peak = max(self.peak, self.in_flight)
        try:
            await asyncio.sleep(self.delay)
            if text == "down" or (text == "flaky" and self.calls[text] == 1):
                raise TransientWorkerError(text)
            if text == "bug":
                raise ValueError("worker bug")
            return text.upper()
        finally:
            self.in_flight -= 1


async def check_ex3():
    async def checks():
        inputs = ["ok-1", "flaky", "ok-2", "down", "ok-3", "bug", "ok-4", "ok-5"]
        w = _StubWorker()
        t0 = time.perf_counter()
        try:
            out = await bounded_fan_out(w, inputs, limit=3)
        except (TransientWorkerError, ValueError) as exc:
            raise AssertionError(f"{type(exc).__name__} escaped bounded_fan_out: handle failures per slot, "
                                 "so one failure can't discard the other results")
        elapsed = time.perf_counter() - t0
        assert isinstance(out, list) and len(out) == len(inputs), "return one (status, value) tuple per input"
        assert [tuple(x) for x in out] == [("ok", "OK-1"), ("ok", "FLAKY"), ("ok", "OK-2"), ("failed", "TransientWorkerError"),
                                           ("ok", "OK-3"), ("failed", "ValueError"), ("ok", "OK-4"), ("ok", "OK-5")], \
            f"wrong results or order: {out}"
        assert w.peak <= 3, f"at most limit=3 calls may be in flight, saw {w.peak}"
        assert w.peak >= 2, "the calls ran one at a time; run them concurrently (asyncio.gather)"
        assert w.calls["flaky"] == 2, "retry a transient failure once"
        assert w.calls["down"] == 2, f"'down' should be tried 1 + retries = 2 times, got {w.calls['down']}"
        assert w.calls["bug"] == 1, "do not retry non-transient errors (ValueError)"
        assert elapsed < 0.7 * sum(w.calls.values()) * w.delay, "too slow: is the work actually concurrent?"
        w0 = _StubWorker()
        out0 = await bounded_fan_out(w0, ["flaky", "ok-1"], limit=2, retries=0)
        assert [tuple(x) for x in out0] == [("failed", "TransientWorkerError"), ("ok", "OK-1")] and w0.calls["flaky"] == 1, \
            "retries=0 means no retry"
        assert await bounded_fan_out(_StubWorker(), [], limit=2) == [], "no inputs -> empty list"
    await _run_check_async(3, checks)


await check_ex3()

### Exercise 4 — A capped evaluator-optimizer loop

Implement `refine(brief, drafter, evaluator, max_rounds=3)` with the Agents SDK. `drafter` and `evaluator` are `Agent`s (the checker gives them scripted models; `evaluator` has `output_type=Review`). [Theory: 8.6](README.md#86-evaluator-optimizer-loops)

- Each round: run the drafter (the first prompt is just the `brief`), then run the evaluator on exactly `f"<brief>{brief}</brief>\n<draft>{draft}</draft>"` (a **fresh context**: the brief and the draft, nothing else), and read the verdict with `final_output_as(Review)`.
- On `"pass"`, return `(draft, review, round_number)` (rounds count from 1).
- On `"revise"`, the next drafter prompt must contain the brief, the previous draft, and every feedback item.
- Never run more than `max_rounds` rounds. At the cap, return the last `(draft, review, max_rounds)`.

In [ ]:
async def refine(brief: str, drafter: Agent, evaluator: Agent, max_rounds: int = 3) -> tuple[str, Review, int]:
    # TODO
    raise NotImplementedError

In [ ]:
def _scripted_pair(verdicts: list[str], max_calls: int = 6) -> tuple[Agent, Agent]:
    """A drafter that numbers its drafts, and an evaluator that returns the scripted verdicts in order."""
    drafts = iter(f"DRAFT-{i}" for i in range(1, 50))
    verdict_iter = iter(verdicts)

    def evaluate(call):
        v = next(verdict_iter)
        fb = [] if v == "pass" else ["FEEDBACK-ALPHA: state the window in UTC", "FEEDBACK-BETA: name the regions"]
        return [say(Review(rationale="r", verdict=v, feedback=fb).model_dump_json())]

    drafter = Agent(name="Drafter", model=policy_model(lambda call: [say(next(drafts))], max_calls))
    evaluator = Agent(name="Evaluator", output_type=Review, model=policy_model(evaluate, max_calls))
    return drafter, evaluator


async def check_ex4():
    async def checks():
        brief = "BRIEF-XYZ: notify customers about the outage."
        d, e = _scripted_pair(["revise", "pass"])
        out = await refine(brief, d, e)
        assert isinstance(out, tuple) and len(out) == 3, "return (draft, review, rounds)"
        draft, review, rounds = out
        assert (draft, rounds) == ("DRAFT-2", 2) and isinstance(review, Review) and review.verdict == "pass", \
            f"revise then pass should return ('DRAFT-2', <pass Review>, 2), got {(draft, review, rounds)}"
        assert user_text(d.model.first_call) == brief, "the first drafter prompt is just the brief"
        second = user_text(d.model.last_call)
        for must in (brief, "DRAFT-1", "FEEDBACK-ALPHA", "FEEDBACK-BETA"):
            assert must in second, f"the revision prompt must contain the brief, the previous draft and every feedback item (missing {must!r})"
        expected_eval = [f"<brief>{brief}</brief>\n<draft>DRAFT-1</draft>", f"<brief>{brief}</brief>\n<draft>DRAFT-2</draft>"]
        got_eval = [user_text(c) for c in e.model.calls]
        assert not any("FEEDBACK" in x for x in got_eval), "the evaluator must not see earlier feedback: give it a fresh context"
        assert got_eval == expected_eval, f"evaluator input must be exactly <brief>…</brief>\\n<draft>…</draft>, got {got_eval}"
        # Fresh context means the WHOLE input is that one message: no earlier turns, no server-side continuation.
        for c, want in zip(e.model.calls, expected_eval):
            items = [{"role": "user", "content": c.input}] if isinstance(c.input, str) else list(c.input)
            got = [(i.get("role"), i.get("content")) if isinstance(i, dict) else repr(i) for i in items]
            assert got == [("user", want)], \
                f"the evaluator must get ONE user message and nothing else (no worker history), got {len(items)} input items"
            assert c.previous_response_id is None and c.conversation_id is None, \
                "the evaluator run must not continue a conversation (previous_response_id / conversation_id)"

        d, e = _scripted_pair(["revise"] * 6)
        draft, review, rounds = await refine(brief, d, e, max_rounds=3)
        assert len(d.model.calls) == 3 and len(e.model.calls) == 3, \
            f"cap at max_rounds=3: drafter ran {len(d.model.calls)}x, evaluator {len(e.model.calls)}x"
        assert (draft, rounds, review.verdict) == ("DRAFT-3", 3, "revise"), "at the cap return the last draft, last review and max_rounds"

        d, e = _scripted_pair(["pass"])
        assert (await refine(brief, d, e))[2] == 1 and len(d.model.calls) == 1, "a first-round pass needs one round"
        d, e = _scripted_pair(["revise"] * 6)
        assert (await refine(brief, d, e, max_rounds=1))[2] == 1 and len(e.model.calls) == 1, "respect max_rounds=1"
    await _run_check_async(4, checks)


await check_ex4()

### Exercise 5 — Independent-reviewer input and duplicate merging

Two functions for the review pipeline. [Theory: 8.9](README.md#89-multi-pass-code-review-and-independent-reviewers)

**(a)** `build_reviewer_input(diff, findings, criteria)` returns the prompt for a **fresh** independent reviewer:

- it contains `<criteria>…</criteria>`, `<diff>…</diff>` and `<findings>…</findings>`, where the text inside `<findings>` is a JSON array;
- the array has every finding, in the original order, with its values unchanged; each keeps **only** `id`, `path`, `line`, `severity`, `rule`, `quote`. Drop everything else the workers produced (their reasoning, transcripts, confidence...), so the reviewer is not anchored by the author's thinking;
- do not mutate `findings`.

**(b)** `merge_findings(findings)` merges duplicates reported by several workers or passes:

- duplicates share the same `(path, line, rule)`;
- a merged finding keeps the first-seen finding's `id` and `quote`, the **most severe** `severity` among the duplicates (`P0` is most severe), and a sorted list `sources` with the ids of every finding merged into it (including its own);
- output dicts have exactly the keys `id`, `path`, `line`, `severity`, `rule`, `quote`, `sources`, sorted by `(SEVERITY_RANK[severity], path, line)`; ties keep first-seen order (Python's `sorted` is stable).

In [ ]:
SEVERITY_RANK = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}
REVIEW_KEYS = ("id", "path", "line", "severity", "rule", "quote")


def build_reviewer_input(diff: str, findings: list[dict], criteria: str) -> str:
    # TODO
    raise NotImplementedError


def merge_findings(findings: list[dict]) -> list[dict]:
    # TODO
    raise NotImplementedError

In [ ]:
_RAW_FINDINGS = [
    {"id": "A1", "path": "app/auth.py", "line": 11, "severity": "P1", "rule": "sql-injection", "quote": "sql = f\"...{name}\"",
     "reasoning": "WORKER-THOUGHT: I think this is bad", "confidence": 0.9},
    {"id": "R1", "path": "app/routes.py", "line": 24, "severity": "P1", "rule": "missing-authz", "quote": "return db.get_user(user_id)",
     "transcript": ["WORKER-THOUGHT: step 1"]},
    {"id": "S7", "path": "app/auth.py", "line": 11, "severity": "P0", "rule": "sql-injection", "quote": "sql = f\"...{name}\""},
    {"id": "T1", "path": "tests/test_routes.py", "line": 6, "severity": "P3", "rule": "weak-test", "quote": "assert ... == 200"},
    {"id": "X1", "path": "app/routes.py", "line": 24, "severity": "P1", "rule": "broken-contract", "quote": "return db.get_user(user_id)"},
    {"id": "R2", "path": "app/routes.py", "line": 24, "severity": "P2", "rule": "missing-authz", "quote": "db.get_user(user_id)"},
]
_DIFF = "+++ app/auth.py\n  11 +     sql = f\"...{name}\"\n+++ app/routes.py\n  24 +     return db.get_user(user_id)"


def check_ex5():
    def checks():
        raw = copy.deepcopy(_RAW_FINDINGS)
        text = build_reviewer_input(_DIFF, raw, "Confirm only what the diff shows.")
        assert isinstance(text, str), "return the prompt as a string"
        assert raw == _RAW_FINDINGS, "do not mutate the findings"
        for tag in ("criteria", "diff", "findings"):
            assert f"<{tag}>" in text and f"</{tag}>" in text, f"missing <{tag}>…</{tag}>"
        assert _DIFF in text and "Confirm only what the diff shows." in text, "include the full diff and the criteria"
        assert "WORKER-THOUGHT" not in text, "the worker's reasoning / transcript leaked into the reviewer input"
        block = re.search(r"<findings>(.*?)</findings>", text, re.S).group(1).strip()
        parsed = json.loads(block)
        assert isinstance(parsed, list) and len(parsed) == len(_RAW_FINDINGS), "<findings> must hold a JSON array with every finding"
        assert all(isinstance(f, dict) and set(f) == set(REVIEW_KEYS) for f in parsed), f"keep only {REVIEW_KEYS} in each finding"
        assert parsed == [{k: f[k] for k in REVIEW_KEYS} for f in _RAW_FINDINGS], \
            "keep every finding's values unchanged and in the original order (the reviewer needs the real ids, lines and quotes)"

        merged = merge_findings(copy.deepcopy(_RAW_FINDINGS))
        assert isinstance(merged, list) and all(isinstance(m, dict) for m in merged), "return a list of dicts"
        assert len(merged) == 4, f"6 findings with 2 duplicate pairs should merge into 4, got {len(merged)}"
        assert all(set(m) == set(REVIEW_KEYS) | {"sources"} for m in merged), "each merged dict needs exactly the review keys + 'sources'"
        sql = next(m for m in merged if m["rule"] == "sql-injection")
        assert sql["severity"] == "P0", "keep the MOST severe severity among duplicates (P0 beats P1)"
        assert sql["id"] == "A1" and sql["sources"] == ["A1", "S7"], "keep the first-seen id and list all merged ids, sorted"
        authz = next(m for m in merged if m["rule"] == "missing-authz")
        assert authz["severity"] == "P1" and authz["sources"] == ["R1", "R2"] and authz["quote"] == "return db.get_user(user_id)", \
            "P1 beats P2; keep the first-seen quote"
        assert [m["id"] for m in merged] == ["A1", "R1", "X1", "T1"], \
            f"sort by (severity rank, path, line): expected A1, R1, X1, T1, got {[m['id'] for m in merged]}"
        assert merge_findings([]) == [], "no findings -> empty list"
    _run_check(5, checks)


check_ex5()

### Exercise 6 (bonus, live — calls the API, ~$0.01) — Run a planner and validate its coverage

Implement `plan_vendor_review(request, model)`: build an `Agent` named `"Planner"` with `output_type=Plan` and the given `model`, whose instructions name **every** facet in `REQUIRED_FACETS` (security, data, cost, operations) and tell it to put every fact a worker needs into each subtask's `context`. Run it on `request` and return the `Plan`. [Theory: 8.5](README.md#85-dynamic-decomposition-orchestrator-workers)

The checker first does an **offline dry run** with a scripted model (it inspects the instructions and the output type), then, if you set `RUN_LIVE_BONUS = True` and have a real key, runs your planner on `MODEL` and validates the plan with your `validate_plan` from exercise 2. This exercise is a bonus and does not count toward the scorecard.

In [ ]:
async def plan_vendor_review(request: str, model) -> Plan:
    # TODO
    raise NotImplementedError

In [ ]:
VENDOR_REQUEST = ("Assess vendor Acme Analytics for our EU customer data. Facts: SOC 2 Type II (2026-03); SAML SSO; data in "
                  "eu-west-1 with 3 EU subprocessors; $48k/year, 3-year term, 7% uplift; 99.9% SLA with 4h P1 response.")


async def check_ex6():
    async def checks():
        good = Plan(subtasks=[Subtask(id=f"s{i}", facet=f, question="q", context="facts")
                              for i, f in enumerate(sorted(REQUIRED_FACETS), 1)])
        scripted = ScriptedModel([[assistant_message(good.model_dump_json())]])
        plan = await plan_vendor_review(VENDOR_REQUEST, scripted)
        assert isinstance(plan, Plan), "return the parsed Plan (final_output_as(Plan))"
        call = scripted.first_call
        assert call is not None, "run the agent with the given model"
        assert call.output_schema is not None and "subtasks" in json.dumps(call.output_schema.json_schema()), "set output_type=Plan"
        missing = [f for f in sorted(REQUIRED_FACETS) if f not in (call.system_instructions or "").lower()]
        assert not missing, f"the instructions must name every required facet, missing {missing}"
        assert "context" in (call.system_instructions or "").lower(), "tell the planner to put the facts into each subtask's context"
        assert VENDOR_REQUEST in user_text(call), "pass the request as the input"
        print("  offline dry run ok")
        if not (RUN_LIVE_BONUS and HAVE_REAL_KEY):
            print("  live part skipped (set RUN_LIVE_BONUS = True in the setup section, with a real OPENAI_API_KEY)")
            return
        live = await plan_vendor_review(VENDOR_REQUEST, MODEL)
        print("  live plan:", [(s.id, s.facet) for s in live.subtasks])
        try:
            report = validate_plan(live, REQUIRED_FACETS)
        except NotImplementedError:
            print("  (implement validate_plan in exercise 2 to validate the live plan)")
            return
        print("  coverage report:", report)
        assert report.ok, "the live plan failed your coverage check: improve the planner instructions and rerun"
    await _run_check_async(6, checks)


await check_ex6()

## Part C — Architecture scenario

A private-equity firm wants an **AI due-diligence assistant**. For each deal, a data room holds about **300 documents** (contracts, financials, HR policies, IP filings, security questionnaires; some are scanned PDFs). The deal team needs a findings memo within **24 hours**, covering six facets: legal, financial, HR, IP, security and operations. Today:

- a partner wants "one big agent with a 1M-token context that reads everything at once";
- another team prototyped **Responses API Multi-agent** and likes that it "just spawns researchers";
- an earlier pilot produced a polished memo that silently skipped IP;
- compliance requires an audit trail of which agent looked at which document, and a cap on spend per deal;
- the firm's document API allows 5 concurrent requests.

Design the orchestration. Cover: the decomposition shape and why; where code vs the model decides; hosted multi-agent vs the Agents SDK for this case; coverage and failure handling; parallelism limits; review and quality gates; testing; tracing and budget. About 10–15 bullet points.

*Write your answer here (double-click to edit).*

<details><summary>Rubric: what a strong answer contains</summary>

- **Shape:** orchestrator-workers inside a code-owned pipeline. A planner (strong model, `output_type=Plan`) maps documents to facet subtasks; code validates, fans out, merges and reviews. Not one 1M-token call: attention dilution and no way to guarantee per-facet depth.
- **Code outside, LLM inside:** code owns stages, caps, gates and failure handling; the model only plans, extracts and judges.
- **Coverage check in code:** every required facet (legal, financial, HR, IP, security, operations) and every document is assigned, or the plan is rejected and re-planned (capped rounds). This is the fix for the skipped-IP pilot, not stronger workers.
- **Workers:** one per document batch and facet, cheap model (`gpt-6-luna`), explicit context per worker, structured findings (`doc_id`, page, quote, severity), each under a verification contract (quote must exist in the document; one retry with the error, then mark the batch `unreviewed`).
- **Parallelism:** `asyncio.gather` behind a semaphore of 5 (the document API limit), `return_exceptions=True`, retries only for transient errors; a missing slot is reported as a gap and blocks a "no issues found" conclusion for that facet. Consider the Batch API or flex processing for parts not needed interactively.
- **Hosted multi-agent:** Responses Multi-agent shares one model and one tool list across the tree and is beta; it cannot give the HR or financial workers least-privilege access or a different model, and the firm needs a deterministic audit trail. Use the Agents SDK; at most use hosted subagents for open-ended exploration where those limits are acceptable.
- **Merge and review:** a synthesizer merges by facet and reports gaps; duplicates merged on document + clause + issue; an independent reviewer (fresh run, different/stronger model, only sources + findings + criteria) confirms or rejects each material finding; optional capped evaluator loop on the memo against an explicit rubric.
- **Human in the loop:** low-confidence or `needs_human` findings and any blocked facet go to the deal team, not into the memo as facts.
- **Testing:** `agents.testing.ScriptedModel` tests for the coverage check, caps, partial failures and budget stop; evals for planner coverage, extraction accuracy and reviewer agreement with human labels.
- **Tracing and audit:** one `trace("Due diligence", group_id=deal_id)` per deal so every planner, worker and reviewer span is grouped; log which worker read which document; note that tracing is unavailable under ZDR and decide what sensitive data spans may contain (`trace_include_sensitive_data`).
- **Budget:** an orchestration-wide budget in code (reserve each run's worst case before it starts, settle `context_wrapper.usage` after, refuse runs that could cross the ceiling); `max_turns` per run, an output-token cap per call and `max_length` on the plan as inner bounds.
- **Avoid:** Agent Builder (shutting down 2026-11-30) and the Assistants API (retired) for new builds.
</details>

## Scorecard

In [ ]:
offline = [EX_RESULTS.get(n, False) for n in range(1, 6)]
exercises = sum(offline) / len(offline)
quiz = globals().get("quiz_score", 0.0)
overall = 0.5 * quiz + 0.5 * exercises
print(f"Quiz:      {quiz:.0%}")
print(f"Exercises: {sum(offline)}/5 passed ({exercises:.0%})  | bonus exercise 6: {'passed' if EX_RESULTS.get(6) else 'not passed'}")
print(f"Overall:   {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%, same bar as the Claude path)")

---

## Solutions (spoilers)

Try everything first. Each block below is collapsed.

<details><summary>Q1 answer</summary>

**A.** The steps are known up front, identical every time, and sequential, so code should own the order: one agent per step, structured outputs as contracts, an `if` as the gate. It is cheaper, deterministic and testable per step. B adds variance and cost for an adaptivity the task does not need. C turns a pipeline into a chain of ownership transfers with no gates. D is a hosted, less controllable option for broad parallel work; the guide says to prefer one agent or a fixed graph when each step depends on the previous one.
</details>

<details><summary>Q2 answer</summary>

**C.** `max_function_tool_concurrency` caps how many local function-tool calls (including nested `as_tool()` agents) the SDK runs at once, so you keep two in flight. A makes the model emit one call per turn, which serializes everything and adds turns. B changes ownership semantics: handoffs cannot merge and only one is honored per turn. D limits turns, not concurrency.
</details>

<details><summary>Q3 answer</summary>

**B.** An evaluator that rejects good drafts on round 1 has a rubric problem: it is too strict, vague, or contradicts what the drafter was told. Calibrate it against human labels and align the criteria. A only makes the loop more expensive. C removes the fresh-context property that makes the evaluator independent. D treats a criteria problem as a model-size problem.
</details>

<details><summary>Q4 answer</summary>

**D.** A worker built with `as_tool()` runs on the tool input only; it never sees the manager's conversation. Pass the facts explicitly, ideally as a structured `parameters=` model so code and reviewers can check them. A does not give the worker the missing facts. B gives it history but moves ownership away from the manager, which then cannot merge. C adds turns, but the worker still has nobody to ask.
</details>

<details><summary>Q5 answer</summary>

**A.** The SDK's type docs and the guide define `max_concurrent_subagents` as the number of subagents that can be active at once across the entire agent tree, including grandchildren and deeper descendants, excluding the root agent. The default is 3. B, C and D misread that scope. (For comparison, the Agents API's own setting defaults to 6 and excludes the coordinator.)
</details>

<details><summary>Q6 answer</summary>

**B.** The Agents API multi-agent guide states that subagents do not support function tools; they inherit configured MCP tools (with their credentials and allowed tools), web search settings and the environment's files and commands. A is not supported. C describes Responses API Multi-agent over WebSocket, a different product. D is not a valid way to route function calls.
</details>

<details><summary>Q7 answer</summary>

**C.** An independent reviewer that sees only the diff and explicit rules removes the author's confirmation bias, and Codex's docs stress that review rules do not replace tests, branch protections or required approvals. A and B keep the generating context. D removes a control the docs say AI review does not replace.
</details>

<details><summary>Q8 answer</summary>

**D.** `max_turns` bounds a single `Runner.run`; dozens of runs can still add up. The OpenAI Agents SDK has no dollar cap, so budget the orchestration in your own code from the usage each result reports. Usage arrives only after a run, so a check made afterwards always overshoots (by a lot with parallel workers); a hard ceiling needs the check before the run starts. A still allows many runs. B describes a Claude Agent SDK option that `RunConfig` does not have. C limits throughput, not spend per job.
</details>

<details><summary>Q9 answer</summary>

**C.** Per-slot handling keeps the good results, a bounded retry handles transient errors, and in a security review a missing file must block approval rather than be silently skipped. A approves an unreviewed file. B throws away eleven good results. D invents findings.
</details>

<details><summary>Q10 answer</summary>

**B.** Git lets a branch be checked out in only one place, and one worktree per writer plus a single merge owner keeps ownership clear. The Codex subagent docs recommend parallel agents mainly for read-heavy work, because parallel writes create conflicts and coordination overhead. A makes collisions more likely. C relies on instructions for a coordination problem. D changes nothing about the conflicts.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
SHAPES = {
    "w1": "single_call",
    "w2": "fixed_pipeline",
    "w3": "routing",
    "w4": "parallelization",
    "w5": "orchestrator_workers",
    "w6": "evaluator_optimizer",
}
```

w4 is sectioning: independent parts, wall-clock time is the slowest worker. w5 is dynamic: the next checks depend on what the first ones reveal. w6 has explicit criteria and feedback that improves the draft.
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def validate_plan(plan: Plan, required_facets: set[str], max_subtasks: int = 6) -> PlanReport:
    covered = {s.facet.strip().lower() for s in plan.subtasks}
    required = {f.strip().lower() for f in required_facets}
    ids = [s.id for s in plan.subtasks]
    return PlanReport(
        missing_facets=sorted(required - covered),
        duplicate_ids=sorted({i for i in ids if ids.count(i) > 1}),
        empty_context_ids=sorted(s.id for s in plan.subtasks if not s.context.strip()),
        too_many=len(plan.subtasks) > max_subtasks,
    )
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
async def bounded_fan_out(worker: Callable[[str], Awaitable[str]], inputs: list[str],
                          limit: int = 3, retries: int = 1) -> list[tuple[str, str]]:
    gate = asyncio.Semaphore(limit)

    async def one(text: str) -> tuple[str, str]:
        for attempt in range(retries + 1):
            try:
                async with gate:                      # hold a slot only while the call runs
                    return ("ok", await worker(text))
            except TransientWorkerError as exc:
                last = exc                            # transient: try again if attempts remain
            except Exception as exc:                  # anything else: fail this slot now
                return ("failed", type(exc).__name__)
        return ("failed", type(last).__name__)

    return list(await asyncio.gather(*(one(t) for t in inputs)))   # order follows inputs
```

Each slot catches its own exceptions, so `gather` never sees one, and one failure cannot discard the others.
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
async def refine(brief: str, drafter: Agent, evaluator: Agent, max_rounds: int = 3) -> tuple[str, Review, int]:
    prompt = brief
    for round_no in range(1, max_rounds + 1):
        draft = (await Runner.run(drafter, prompt)).final_output
        review = (await Runner.run(evaluator, f"<brief>{brief}</brief>\n<draft>{draft}</draft>")).final_output_as(Review)
        if review.verdict == "pass":
            return draft, review, round_no
        prompt = (f"<brief>{brief}</brief>\n<previous_draft>{draft}</previous_draft>\n<feedback>\n"
                  + "\n".join(f"- {item}" for item in review.feedback) + "\n</feedback>")
    return draft, review, max_rounds
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def build_reviewer_input(diff: str, findings: list[dict], criteria: str) -> str:
    clean = [{k: f[k] for k in REVIEW_KEYS} for f in findings]      # no worker reasoning, no transcripts
    return (f"<criteria>\n{criteria}\n</criteria>\n"
            f"<diff>\n{diff}\n</diff>\n"
            f"<findings>\n{json.dumps(clean, indent=2)}\n</findings>")


def merge_findings(findings: list[dict]) -> list[dict]:
    merged: dict[tuple, dict] = {}
    for f in findings:
        key = (f["path"], f["line"], f["rule"])
        if key not in merged:
            merged[key] = {**{k: f[k] for k in REVIEW_KEYS}, "sources": [f["id"]]}
            continue
        m = merged[key]
        m["sources"].append(f["id"])
        if SEVERITY_RANK[f["severity"]] < SEVERITY_RANK[m["severity"]]:
            m["severity"] = f["severity"]
    for m in merged.values():
        m["sources"].sort()
    return sorted(merged.values(), key=lambda m: (SEVERITY_RANK[m["severity"]], m["path"], m["line"]))
```
</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
async def plan_vendor_review(request: str, model) -> Plan:
    planner = Agent(
        name="Planner",
        model=model,
        output_type=Plan,
        instructions=("Split the vendor assessment into independent subtasks, at least one per facet: "
                      "security, data, cost, operations. Set each subtask's facet to one of those words. "
                      "Put every fact a worker needs into its context; workers see nothing else."),
    )
    return (await Runner.run(planner, request)).final_output_as(Plan)
```
</details>

## Next chapter

Next: [Chapter 09 — Escalation and human-in-the-loop](../09-escalation-human-in-the-loop/README.md), where the `needs_human` and `unreviewed` outcomes from this chapter become a designed escalation path.